# Build the DRAAD AI components

**Participant lab · 60–90 minutes · Level 200**

The Next.js interface, FastAPI/SSE pipeline, synthetic data and deterministic safety gates are supplied. Your team implements and deploys the four prompt agents consumed by that runtime.

Use only the synthetic workshop environment. This notebook is intentionally incomplete: each `TODO` is learner-owned code.

## Completion contract

By the end, you must show:

1. four new agent version numbers;
2. one Search-grounded Q&A response with an inspectable source;
3. one retriever response matching the documented JSON contract; and
4. one complete incident result with five deterministic rule verdicts and a human-explainable final action.

Read [`../PLAYBOOK.md`](../PLAYBOOK.md) before starting.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AISearchIndexResource,
    AzureAISearchQueryType,
    AzureAISearchTool,
    AzureAISearchToolResource,
    PromptAgentDefinition,
)
from azure.core.exceptions import HttpResponseError
from azure.identity import InteractiveBrowserCredential
from openai import APIStatusError


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / '.env.example').is_file() and (candidate / 'app').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the workshop repository.')


ROOT = find_repository_root()
load_dotenv(ROOT / '.env')
sys.path.insert(0, str(ROOT / 'app' / 'backend'))

from agent_names import MATCHER_NAME, QA_NAME, RETRIEVER_NAME, REVIEWER_NAME

required = (
    'FOUNDRY_PROJECT_ENDPOINT',
    'FOUNDRY_MODEL',
    'AZURE_SEARCH_CONNECTION_NAME',
    'AZURE_SEARCH_INDEX',
    'WORKSHOP_RESOURCE_NAMESPACE',
)
missing = [name for name in required if not os.getenv(name)]
assert not missing, f'Missing environment variables: {missing}'

credential = InteractiveBrowserCredential()
project = AIProjectClient(
    endpoint=os.environ['FOUNDRY_PROJECT_ENDPOINT'],
    credential=credential,
)
print({
    'namespace': os.environ['WORKSHOP_RESOURCE_NAMESPACE'],
    'model': os.environ['FOUNDRY_MODEL'],
    'agent_names': [QA_NAME, RETRIEVER_NAME, MATCHER_NAME, REVIEWER_NAME],
})


In [ ]:
def with_project_retry(label: str, operation, attempts: int = 6):
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except HttpResponseError as exc:
            transient = exc.status_code == 404 and 'project not found' in str(exc).lower()
            if not transient or attempt == attempts:
                raise
            delay = min(5 * (2 ** (attempt - 1)), 30)
            print(f'{label}: project data plane is still propagating; retrying in {delay}s')
            time.sleep(delay)


def with_agent_retry(label: str, operation, attempts: int = 6):
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except (HttpResponseError, APIStatusError) as exc:
            transient = getattr(exc, 'status_code', None) in {404, 409, 429, 500, 502, 503, 504}
            if not transient or attempt == attempts:
                raise
            delay = min(3 * (2 ** (attempt - 1)), 20)
            print(f'{label}: agent version is still propagating; retrying in {delay}s')
            time.sleep(delay)


def resolve_connection_id(connection_name: str, attempts: int = 6) -> str:
    for attempt in range(1, attempts + 1):
        try:
            matches = [
                item for item in project.connections.list()
                if item.name == connection_name
            ]
        except HttpResponseError as exc:
            transient = exc.status_code == 404 and 'project not found' in str(exc).lower()
            if not transient or attempt == attempts:
                raise
            matches = []
        if len(matches) == 1:
            return matches[0].id
        if len(matches) > 1:
            raise RuntimeError(
                f'Expected one project connection named {connection_name!r}; found {len(matches)}.'
            )
        if attempt == attempts:
            raise RuntimeError(
                f'Project connection {connection_name!r} was not visible after {attempts} attempts.'
            )
        delay = min(5 * (2 ** (attempt - 1)), 30)
        print(f'Resolve Search connection: not visible yet; retrying in {delay}s')
        time.sleep(delay)


def require_prompt(label: str, value: str | None, required_terms: tuple[str, ...]) -> str:
    if not isinstance(value, str) or len(value.strip()) < 80:
        raise AssertionError(f'{label} instructions are still incomplete.')
    lowered = value.lower()
    missing_terms = [term for term in required_terms if term.lower() not in lowered]
    if missing_terms:
        raise AssertionError(f'{label} instructions are missing contract terms: {missing_terms}')
    return value.strip()


def deploy_prompt_agent(name: str, instructions: str, tools: list | None = None):
    return with_project_retry(
        f'Deploy {name}',
        lambda: project.agents.create_version(
            agent_name=name,
            definition=PromptAgentDefinition(
                model=os.environ['FOUNDRY_MODEL'],
                instructions=instructions,
                tools=tools or None,
            ),
            description='Participant-built DRAAD workshop agent.',
        ),
    )


SEARCH_CONNECTION_ID = resolve_connection_id(os.environ['AZURE_SEARCH_CONNECTION_NAME'])
print('Search connection resolved:', SEARCH_CONNECTION_ID)

## Participant task 1 — Search tool and Q&A agent

Implement one Azure AI Search tool targeting the prepared VWI index. Use semantic querying and a bounded `top_k`.

Write Q&A instructions that:

- answer only from retrieved evidence;
- preserve the native Search citation annotation and finish with a literal `Sources:` list whose entries can be inspected;
- distinguish procedure evidence from unsupported claims; and
- say that evidence is insufficient instead of guessing.

In [ ]:
def build_search_tool(connection_id: str, index_name: str) -> AzureAISearchTool:
    # TODO 1A: Return AzureAISearchTool with one AISearchIndexResource.
    # Required parameters: project_connection_id, index_name, semantic query type, bounded top_k.
    raise NotImplementedError('Implement the Azure AI Search tool.')


QA_INSTRUCTIONS = None  # TODO 1B: Write the grounded Q&A contract.

search_tool = build_search_tool(SEARCH_CONNECTION_ID, os.environ['AZURE_SEARCH_INDEX'])
QA_INSTRUCTIONS = require_prompt(
    'Q&A', QA_INSTRUCTIONS, ('source', 'insufficient'),
)
print('Task 1 contract ready.')

## Participant task 2 — Procedure retriever

Write instructions for a broad retriever, not a decision maker. It must copy identifiers from Search and return JSON only:

```json
{"vwi_candidates": [{"vwi_id": "E-85", "title": "...", "content": "...", "source_doc": "..."}]}
```

Do not let this agent choose a crew, authorization scope or operational action.

In [ ]:
RETRIEVER_INSTRUCTIONS = None  # TODO 2: Write the JSON-only retrieval contract.

RETRIEVER_INSTRUCTIONS = require_prompt(
    'Retriever',
    RETRIEVER_INSTRUCTIONS,
    ('vwi_candidates', 'vwi_id', 'content', 'source_doc'),
)
print('Task 2 contract ready.')

## Participant task 3 — Dispatch matcher

The matcher receives the original incident, retrieved VWI candidates and authorization scopes already filtered by deterministic Python.

It must leave `matched_crew`, `matched_raamopdracht_id`, `coverage_status` and `operational_action` unset for Python. It must never invent a VWI identifier and must refuse out-of-scope incidents. Return JSON only with this exact interface:

```json
{
  "incident_id": "<input id or null>",
  "vwis": [{"vwi_id": "E-85", "confidence": "confirmed|candidate"}],
  "matched_crew": null,
  "matched_raamopdracht_id": null,
  "coverage_status": null,
  "review_status": "pass",
  "operational_action": null,
  "rationale": "...",
  "citations": {
    "vwi_refs": ["E-85"],
    "raamopdracht_scope_excerpts": ["<verbatim scope excerpt>"],
    "bei_rule_refs": []
  }
}
```

In [ ]:
MATCHER_INSTRUCTIONS = None  # TODO 3: Write the matcher input/output and refusal contract.

MATCHER_INSTRUCTIONS = require_prompt(
    'Matcher',
    MATCHER_INSTRUCTIONS,
    ('vwis', 'confidence', 'matched_crew', 'matched_raamopdracht_id', 'citations'),
)
print('Task 3 contract ready.')

## Participant task 4 — Dispatch reviewer

The reviewer challenges the proposal against the original incident and deterministic findings. A `revise` result requires concrete `feedback_for_matcher`; malformed output must be safe to escalate. Return JSON only with this exact interface:

```json
{
  "review_status": "pass|revise|flagged_for_human_review",
  "findings": [
    {"criterion": "selection_appropriateness|dangerous_situation|symptom_vs_cause|escalation_appropriateness", "verdict": "pass|fail", "reason": "..."}
  ],
  "feedback_for_matcher": "<concrete correction or null>"
}
```

The reviewer cannot override failed deterministic rules.

In [ ]:
REVIEWER_INSTRUCTIONS = None  # TODO 4: Write the reviewer and escalation contract.

REVIEWER_INSTRUCTIONS = require_prompt(
    'Reviewer',
    REVIEWER_INSTRUCTIONS,
    ('pass', 'revise', 'flagged_for_human_review', 'findings', 'feedback_for_matcher'),
)
print('Task 4 contract ready.')

## Deploy the participant implementations

This creates new versions under your team's scoped names. The Search tool belongs only on the Q&A and retriever agents. Record the four version numbers as evidence.

In [ ]:
agent_specs = (
    (QA_NAME, QA_INSTRUCTIONS, [search_tool]),
    (RETRIEVER_NAME, RETRIEVER_INSTRUCTIONS, [search_tool]),
    (MATCHER_NAME, MATCHER_INSTRUCTIONS, None),
    (REVIEWER_NAME, REVIEWER_INSTRUCTIONS, None),
)

deployed = []
for name, instructions, tools in agent_specs:
    version = deploy_prompt_agent(name, instructions, tools)
    deployed.append({'name': version.name, 'version': version.version})

assert len(deployed) == 4
deployed

## Smoke test 1 — grounded Q&A

Ask a question answered by a known synthetic VWI. Inspect the source reference and verify it points to evidence in the prepared Search index. A fluent answer without inspectable evidence does not pass.

In [ ]:
qa_client = project.get_openai_client(agent_name=QA_NAME)
qa_response = with_agent_retry(
    'Q&A smoke test',
    lambda: qa_client.responses.create(
        input='What does synthetic VWI E-85 cover? Cite the indexed source.',
    ),
)
assert qa_response.output_text.strip(), 'Q&A agent returned no text.'
assert '†source' in qa_response.output_text, 'Q&A answer has no Search citation annotation.'
assert 'Sources:' in qa_response.output_text, 'Q&A answer has no source list.'
print(qa_response.output_text)
print('Manual gate: identify and inspect the cited Search source before continuing.')

## Smoke test 2 — retriever contract

Run the retriever directly and prove that its output is JSON with a non-empty `vwi_candidates` list.

In [ ]:
retriever_client = project.get_openai_client(agent_name=RETRIEVER_NAME)
retriever_response = with_agent_retry(
    'Retriever smoke test',
    lambda: retriever_client.responses.create(
        input='Synthetic LS incident: suspected blown fuse in a service cabinet.',
    ),
)
retriever_payload = json.loads(retriever_response.output_text)
candidates = retriever_payload.get('vwi_candidates')
assert isinstance(candidates, list) and candidates, retriever_payload
assert all(item.get('vwi_id') and item.get('source_doc') for item in candidates)
print(json.dumps(retriever_payload, indent=2, ensure_ascii=False))

## Smoke test 3 — complete application pipeline

Run one versioned synthetic incident through the same Python pipeline used by FastAPI. The supplied deterministic code must still own rules, authorization/crew selection and the final action.

In [ ]:
from pipeline import run_chat

incident_data = json.loads((ROOT / 'app' / 'data' / 'incidents.json').read_text(encoding='utf-8'))
incident = incident_data['incidents'][1]
result = await run_chat(json.dumps(incident, ensure_ascii=False))

assert result.get('type') == 'dispatch', result
dispatch = result.get('response')
assert isinstance(dispatch, dict), result
assert len(dispatch.get('rule_verdicts', [])) == 5, dispatch
assert dispatch.get('operational_action') in {'dispatch_ok', 'wv_escalation_needed'}
print(json.dumps(dispatch, indent=2, ensure_ascii=False))

## Handoff

Record the four version numbers, Q&A evidence, retriever JSON, final rule verdicts and one correction your team made. Do not include `.env`, tokens, personal data or real operational records.

The agent versions are intentionally retained for the application demonstration. Cleanup remains namespace-scoped and facilitator-controlled.